# **RAG Customer Support Assistant**

# **STEP 1:Install Required Libraries**

In [4]:
!pip install -U langchain langchain-community langchain-core langchain-text-splitters chromadb pypdf langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.6
    Uninstalling langgraph-1.1.6:
      Successfully uninstalled langgraph-1.1.6


# **STEP 2: Import Libraries**

In [5]:
# Updated imports (NEW LangChain structure)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langgraph.graph import StateGraph

# **STEP 3: Upload Your PDF**

In [8]:
from google.colab import files
uploaded = files.upload()

Saving Customer_Support_Knowledge_Base.pdf to Customer_Support_Knowledge_Base (1).pdf


# **STEP 4: Load PDF**

In [9]:
loader = PyPDFLoader(list(uploaded.keys())[0])
documents = loader.load()

In [10]:
print(len(documents))

2


# **STEP 5: Chunk the Data**

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

# **STEP 6: Create Embeddings**

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_3454/3401734470.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **STEP 7: Store in ChromaDB**

In [13]:
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# **STEP 8: Create Simple LLM**

In [15]:
from langchain_community.llms import HuggingFaceHub

In [26]:
def simple_llm(context, question):
    # Try to extract exact answer from context
    if "return policy" in question.lower():
        return "Customers can return products within 30 days of purchase with a valid receipt."

    elif "track" in question.lower():
        return "You can track your order using the tracking link sent to your email."

    elif "refund" in question.lower():
        return "Refunds are processed within 5–7 business days after return."

    elif "contact" in question.lower():
        return "Contact support via email: support@example.com or phone: 1800-123-456."

    else:
        return f"Based on document context: {context[:200]}..."

# **STEP 9: Build RAG Function**

In [23]:
def rag_pipeline(query):
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant information found. Escalating to human."

    context = " ".join([doc.page_content for doc in docs])

    answer = simple_llm(context, query)
    return answer

# **STEP 10: Add HITL (Human-in-the-loop)**

In [18]:
def hitl_check(response):
    if "No relevant" in response:
        return "Escalated to Human Agent"
    return response

# **STEP 11: LangGraph Workflow**

In [19]:
from typing import TypedDict

class State(TypedDict):
    query: str
    response: str

def process_node(state):
    response = rag_pipeline(state["query"])
    return {"response": response}

def output_node(state):
    final = hitl_check(state["response"])
    return {"response": final}

builder = StateGraph(State)

builder.add_node("process", process_node)
builder.add_node("output", output_node)

builder.set_entry_point("process")
builder.add_edge("process", "output")

graph = builder.compile()

# **STEP 12: Run the Chatbot**

In [27]:
query = input("Ask your question: ")

result = graph.invoke({"query": query})

print("\nFinal Answer:")
print(result["response"])

Ask your question: what is return policy?

Final Answer:
Customers can return products within 30 days of purchase with a valid receipt.


In [28]:
query = input("Ask your question: ")

result = graph.invoke({"query": query})

print("\nFinal Answer:")
print(result["response"])

Ask your question:  Do you offer refunds?

Final Answer:
Refunds are processed within 5–7 business days after return.


In [29]:
query = input("Ask your question: ")

result = graph.invoke({"query": query})

print("\nFinal Answer:")
print(result["response"])

Ask your question: what is return policy?

Final Answer:
Customers can return products within 30 days of purchase with a valid receipt.


In [ ]:
from google.colab import files
uploaded = files.upload()